# 02 · A robots.txt parser following RFC 9309

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/pretraining-data/02_robots_txt.ipynb) &nbsp; 📖 Concept deck: [robots.txt and consent](https://nilesh-salpe.github.io/visual-ai/pretraining-data/04-robots-and-consent/) &nbsp; ⏱️ Suggested time: **35 min**

**You will build:** parsing robots.txt into user-agent groups, matching paths with `*` and `$`, the longest-match rule (allow wins ties), and the status-code rules for missing or failing robots.txt.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


import re
EXAMPLE = """
# example
User-agent: *
Disallow: /private/
Allow: /private/public-page.html

User-agent: ExampleBot
User-agent: OtherBot
Disallow: /tmp
Disallow: /*.pdf$
Allow: /tmp/ok

User-agent: CCBot
Disallow: /
"""

## Task 1: Parse groups

Return a dict `{product_token_lowercase: [(kind, path), ...]}` where kind is `'allow'` or `'disallow'`. Consecutive `user-agent` lines share the rules that follow them; a new `user-agent` line after rules starts a new group. Strip `#` comments, ignore case of field names, and ignore rules before the first user-agent line. Empty `Disallow:` lines mean "allow everything" and can be skipped.

In [ ]:
def parse_robots(text):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_parse_robots(f):
    g = f(EXAMPLE)
    assert set(g) == {"*", "examplebot", "otherbot", "ccbot"}
    assert g["examplebot"] == g["otherbot"], "two user-agent lines share one group"
    assert ("allow", "/private/public-page.html") in g["*"]
    assert g["ccbot"] == [("disallow", "/")]

check("parse_robots", _test_parse_robots, parse_robots)

## Task 2: Path patterns with * and $

Return the **length of the match** (number of pattern characters, excluding `$`, matched against the path prefix) if `pattern` matches `path`, else `-1`. `*` matches any sequence of characters; a trailing `$` anchors the end. Matching starts at the first character of the path. Hint: translate the pattern to a regular expression.

In [ ]:
def rule_match(pattern, path):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_rule_match(f):
    assert f("/tmp", "/tmp/a") == 4 and f("/tmp", "/tm") == -1
    assert f("/*.pdf$", "/docs/x.pdf") > 0 and f("/*.pdf$", "/docs/x.pdf?v=1") == -1
    assert f("/", "/anything") == 1

check("rule_match", _test_rule_match, rule_match)

## Task 3: Is this URL allowed?

`allowed(groups, agent, path)`: pick the group for the agent's product token (case-insensitive), else the `*` group, else everything is allowed. Among matching rules, the **longest match wins**; if an allow and a disallow match equally long, **allow** wins. `/robots.txt` itself is always allowed.

In [ ]:
def allowed(groups, agent, path):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_allowed(f):
    g = {"*": [("disallow", "/private/"), ("allow", "/private/public-page.html")],
         "examplebot": [("disallow", "/tmp"), ("disallow", "/*.pdf$"), ("allow", "/tmp/ok")],
         "otherbot": [("disallow", "/tmp"), ("disallow", "/*.pdf$"), ("allow", "/tmp/ok")], "ccbot": [("disallow", "/")]}
    assert f(g, "SomeBot", "/private/x") is False and f(g, "SomeBot", "/private/public-page.html") is True, "longer allow beats shorter disallow"
    assert f(g, "examplebot", "/tmp/ok/1") is True and f(g, "ExampleBot", "/tmp/no") is False
    assert f(g, "ExampleBot", "/a/b.pdf") is False and f(g, "ExampleBot", "/private/x") is True, "a specific group replaces the * group entirely"
    assert f(g, "CCBot", "/anything") is False and f(g, "CCBot", "/robots.txt") is True
    assert f({"*": [("disallow", "/a"), ("allow", "/a")]}, "x", "/a") is True, "ties go to allow"

check("allowed", _test_allowed, allowed)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Parse groups
def parse_robots_solution(text):
    groups, current, in_rules = {}, [], False
    for line in text.splitlines():
        line = line.split("#", 1)[0].strip()
        if ":" not in line: continue
        k, v = (x.strip() for x in line.split(":", 1)); k = k.lower()
        if k == "user-agent":
            if in_rules: current, in_rules = [], False
            current.append(v.lower()); groups.setdefault(v.lower(), [])
        elif k in ("allow", "disallow") and current:
            in_rules = True
            if v:
                for ua in current: groups[ua].append((k, v))
    return groups

check("parse_robots (reference)", _test_parse_robots, parse_robots_solution)

In [ ]:
# @title Solution for Task 2: Path patterns with * and $
def rule_match_solution(pattern, path):
    anchored = pattern.endswith("$"); p = pattern[:-1] if anchored else pattern
    rx = "^" + ".*".join(re.escape(part) for part in p.split("*")) + ("$" if anchored else "")
    return len(p) if re.match(rx, path) else -1

check("rule_match (reference)", _test_rule_match, rule_match_solution)

In [ ]:
# @title Solution for Task 3: Is this URL allowed?
def allowed_solution(groups, agent, path):
    if path == "/robots.txt": return True
    rules = groups.get(agent.lower(), groups.get("*"))
    if not rules: return True
    best_len, best_kind = -1, "allow"
    for kind, pat in rules:
        n = rule_match_solution(pat, path)
        if n > best_len or (n == best_len and kind == "allow"):
            best_len, best_kind = n, kind
    return best_len < 0 or best_kind == "allow"

check("allowed (reference)", _test_allowed, allowed_solution)

---
## Experiments
**What to do when robots.txt cannot be fetched (RFC 9309, §2.3.1).** A 4xx status means "unavailable": the crawler MAY access anything. A 5xx or network error means "unreachable": assume **complete disallow**. Caches should not be used for more than 24 hours, and parsers must accept at least 500 KiB.

In [ ]:
def policy(status):
    if 200 <= status < 300: return "parse and obey the file"
    if 400 <= status < 500: return "unavailable → may crawl everything"
    if 500 <= status < 600: return "unreachable → assume complete disallow"
    return "follow redirects (at least five)"
for s in [200, 301, 404, 403, 500, 503]: print(s, "→", policy(s))

---
## Takeaways
* Groups are chosen by product token (case-insensitive); a matching group replaces the `*` group.
* Longest match wins; on a tie, allow wins; `*` and `$` are the only special characters.
* 4xx → crawl allowed; 5xx/unreachable → crawl nothing. Crawl-delay is a non-standard extension (Common Crawl's CCBot obeys it).